In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, avg, isnan, round

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Αρχικοποίηση SparkSession
spark = SparkSession.builder \
    .appName("TelecomChurnCleaning") \
    .getOrCreate()

# Φόρτωση δεδομένων
df = spark.read.csv("telecom_churn_10k.csv", header=True, inferSchema=True)

# Εμφάνιση στοιχείων
print("Πρώτες 10 γραμμές:")
df.show(10)
print("Σχήμα Δεδομένων (Schema):")
df.printSchema()
print(f"Συνολικός αριθμός εγγραφών: {df.count()}")
missing_values = df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns])
missing_values.show()

df_cleaned = df.filter(col("CHURN").isNotNull())

stats = df_cleaned.select(
    round(avg("AGE"),2).alias("avg_age"),
    round(avg("TENURE_MONTHS"),2).alias("avg_tenure"),
    round(avg("MONTHLY_CHARGES"),2).alias("avg_monthly"),
    round(avg("TOTAL_CHARGES"),2).alias("avg_total")
).collect()[0]

df_cleaned.show(10)

df_cleaned = df_cleaned.fillna({
    "AGE" : stats["avg_age"],
    "TENURE_MONTHS" : stats["avg_tenure"],
    "MONTHLY_CHARGES" : stats["avg_monthly"],
    "TOTAL_CHARGES" : stats["avg_total"]
})

df_cleaned.show(10)

df_cleaned.select("AGE", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES").describe().show()

df_cleaned.groupBy("CHURN").agg(
    avg("TENURE_MONTHS").alias("Avg_Tenure"),
    avg("MONTHLY_CHARGES").alias("Avg_Monthly_Charges")
).show()

df_final = df_cleaned.withColumn("NUM_SERVICES", col("HAS_INTERNET") + col("HAS_MOBILE") + col("HAS_TV")).withColumn("IS_LONG_TENURE", when(col("TENURE_MONTHS") >=24,1).otherwise(0)).withColumn("COMPLAINTS_PER_MONTH", when(col("TENURE_MONTHS") > 0, col("NUM_COMPLAINTS") / col("TENURE_MONTHS")).otherwise(col("NUM_COMPLAINTS")))

df_final.select("AGE", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES", "NUM_SERVICES", "IS_LONG_TENURE", "COMPLAINTS_PER_MONTH").show(10)

df_final.createOrReplaceTempView("churn_view")

spark.sql("""
SELECT * FROM churn_view;
""").show(10)

spark.sql("""
SELECT
    CONTRACT_TYPE,
    COUNT(*) AS TOTAL_CUSTOMERS,
    SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) AS TOTAL_CHURNED,
    ROUND(100.0 * SUM(CASE WHEN CHURN = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS Churn_Percentage
FROM churn_view
GROUP BY CONTRACT_TYPE
ORDER BY Churn_Percentage DESC
""").show(10)

spark.sql("""
SELECT 
    NUM_SERVICES, 
    ROUND(100.0 * SUM(CHURN) / COUNT(*), 2) AS Churn_Percentage
FROM churn_view
GROUP BY NUM_SERVICES  
ORDER BY NUM_SERVICES
""").show(10)

spark.sql("""
    SELECT 
        CHURN,
        CONTRACT_TYPE,
        ROUND(AVG(MONTHLY_CHARGES), 2) as Avg_Monthly_Charges
    FROM churn_view
    GROUP BY CHURN, CONTRACT_TYPE
    ORDER BY CONTRACT_TYPE, CHURN
""").show()

top_countries_df = spark.sql("""
SELECT
    COUNTRY,
    COUNT(*) AS Customers,
    ROUND(100.0 * SUM(CHURN) / COUNT(*), 2) AS Churn_Percentage
FROM churn_view
GROUP BY COUNTRY
HAVING COUNT(*) >= 100
ORDER BY Churn_Percentage DESC
LIMIT 5
""").show(10)

selected_cols = ["AGE", "TENURE_MONTHS", "NUM_COMPLAINTS", "SUPPORT_CALLS", 
                 "HAS_INTERNET", "HAS_MOBILE", "HAS_TV", "NUM_SERVICES",
                 "CONTRACT_TYPE", "PAYMENT_METHOD", "COUNTRY", "MONTHLY_CHARGES","COMPLAINTS_PER_MONTH"]

df_model = df_final.select(selected_cols)


# 1. Μετατροπή κατηγορικών κειμένων σε νούμερα (Indexing)
indexers = [StringIndexer(inputCol=c, outputCol=c+"_index").setHandleInvalid("keep") 
            for c in ["CONTRACT_TYPE", "PAYMENT_METHOD", "COUNTRY"]]

# 2. One-Hot Encoding (Μετατρέπει τα indexes σε δυαδικές στήλες - προαιρετικό αλλά βελτιώνει την απόδοση)
encoders = [OneHotEncoder(inputCol=c+"_index", outputCol=c+"_vec") 
            for c in ["CONTRACT_TYPE", "PAYMENT_METHOD", "COUNTRY"]]

# 3. Συγκέντρωση όλων των features σε ένα διάνυσμα (Vector)
feature_cols = ["AGE", "TENURE_MONTHS", "NUM_COMPLAINTS", "SUPPORT_CALLS", 
                "HAS_INTERNET", "HAS_MOBILE", "HAS_TV", "NUM_SERVICES",
                "CONTRACT_TYPE_vec", "PAYMENT_METHOD_vec", "COUNTRY_vec"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# 4. Ορισμός του Μοντέλου (Decision Tree)
dt = DecisionTreeRegressor(labelCol="MONTHLY_CHARGES", featuresCol="features")

# 5. Δημιουργία του Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, dt])


# Train/Test Split (70-30)
train_data, test_data = df_model.randomSplit([0.7, 0.3], seed=42)

# Εκπαίδευση του Pipeline
model = pipeline.fit(train_data)

# Προβλέψεις στο test set
predictions = model.transform(test_data)

# Εμφάνιση δειγμάτων πρόβλεψης
predictions.select("MONTHLY_CHARGES", "prediction", "NUM_COMPLAINTS").show(10)

# Αξιολόγηση με RMSE και R2
evaluator_rmse = RegressionEvaluator(labelCol="MONTHLY_CHARGES", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="MONTHLY_CHARGES", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

# Αποθήκευση του μοντέλου στον φάκελο που "βλέπει" και το API
model.write().overwrite().save("models/monthly_charges_model")
print("Το μοντέλο αποθηκεύτηκε επιτυχώς!")

Πρώτες 10 γραμμές:
+-----------+------+----+-------+-------------+--------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|CUSTOMER_ID|GENDER| AGE|COUNTRY|TENURE_MONTHS| CONTRACT_TYPE|HAS_INTERNET|HAS_MOBILE|HAS_TV|MONTHLY_CHARGES|TOTAL_CHARGES|NUM_COMPLAINTS|SUPPORT_CALLS|PAYMENT_METHOD|CHURN|
+-----------+------+----+-------+-------------+--------------+------------+----------+------+---------------+-------------+--------------+-------------+--------------+-----+
|    C000001|  Male|68.0|     FR|         61.0|Month-to-month|           0|         1|     0|          17.54|      1077.63|             0|            0| Bank transfer|  1.0|
|    C000002|Female|65.0|     FR|         NULL|Month-to-month|           1|         1|     0|          50.82|       833.94|             1|            1|   Credit card|  0.0|
|    C000003|  Male|36.0|     FR|         53.0|      Two year|           1|         1|     0|          34.44|  